# 02 - Fixed-EOF NAM and AO

Run from `code/02_diagnostics` on STREAM2. Archived MLS inputs below `PAPER1_ARCHIVE_ROOT` are read-only; staged scientific inputs come from `PAPER1_PREPROCESSED_ROOT`; every new product is schema-validated and atomically written below `PAPER1_DERIVED_ROOT` (default: repository-local `Paper1/runtime`; overrides must resolve to that runtime directory or one of its descendants).


## Shared pressure interpolation and schema

All sources use the same 23-level hPa grid. NAM products always contain `nam` and `ao`; AO is the exact 1000 hPa slice. Integer `date` is retained for every restart case.

Inputs: The read-only archive or schema-validated staging paths explicitly opened in the following code cell.

Outputs: The in-memory object(s) and atomic staging product(s) explicitly named in the following code cell; setup-only cells emit no file.

Method: Apply the scientific definition stated above, enforce its declared dimensions/counts/parameters, validate any temporary output, then atomically install it without modifying a source file.


In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

def discover_diagnostic_directory():
    candidates = (
        Path.cwd(), Path.cwd() / "02_diagnostics",
        Path.cwd() / "Paper1" / "02_diagnostics",
        Path.cwd() / "code_cleaned" / "Paper1" / "02_diagnostics",
    )
    for candidate in candidates:
        if (candidate / "lib" / "workflow_io.py").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Cannot locate Paper1/02_diagnostics/lib from the current working directory"
    )

NOTEBOOK_DIR = discover_diagnostic_directory()
LIB = NOTEBOOK_DIR / "lib"
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))

from workflow_io import (
    PRODUCT_VERSION, archive_root, derived_root, preprocessed_root, product_path,
    write_csv_atomic, write_netcdf_atomic,
)

ARCHIVE_ROOT = archive_root()
PREPROCESSED_ROOT = preprocessed_root()
DERIVED_ROOT = derived_root()
MARINA_ROOT = Path(os.environ.get(
    "PAPER1_MARINA_ROOT",
    "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
    "BWCN.e122.f19_g16.002_0008/Mar",
))
OVERWRITE = os.environ.get("PAPER1_OVERWRITE_STAGING", "0") == "1"
print("read-only archive root:", ARCHIVE_ROOT)
print("preprocessed staging input root:", PREPROCESSED_ROOT)
print("read-only Marina March root:", MARINA_ROOT)
print("staging output root:", DERIVED_ROOT)
print("diagnostic notebook directory:", NOTEBOOK_DIR)

from paper1_diagnostics import (
    PLEV_HPA, date_int, hybrid_mid_pressure, log_pressure_interpolate,
    parse_member, parse_year, project_nam, select_pressure_levels,
    train_fixed_nam_reference,
)

def pressure_zonal_z3(path, *, hybrid):
    with xr.open_dataset(path, decode_times=True, chunks={"time": 12}) as source:
        if hybrid:
            height = log_pressure_interpolate(
                source["Z3"], hybrid_mid_pressure(source), PLEV_HPA
            )
        else:
            height = select_pressure_levels(source["Z3"], PLEV_HPA)
        if "lon" in height.dims:
            height = height.mean("lon", skipna=True)
        height = height.transpose("time", "plev", "lat").load()
        dates = date_int(source)
    return height.assign_coords(date=("time", dates))

def write_nam(dataset, name, member_product=False):
    dataset.plev.attrs.update(units="hPa", positive="down")
    required = {
        "nam": (("member", "time", "plev") if member_product else ("time", "plev")),
        "ao": (("member", "time") if member_product else ("time",)),
    }
    write_netcdf_atomic(
        dataset, product_path("nam", name), required_vars=required,
        required_coords=("date", "plev") + (("member",) if member_product else ()),
        exact_sizes=({"member": 30} if member_product else None), overwrite=OVERWRITE,
    )


## MERRA-2 reference and projection

Uses daily Z3 for 1980--2025. Calendar-month zonal-mean anomalies from 1980--2000 train the fixed leading EOF. Daily anomalies use a truly month-day/no-leap circular centered 21-day climatology.

Inputs: The read-only archive or schema-validated staging paths explicitly opened in the following code cell.

Outputs: The in-memory object(s) and atomic staging product(s) explicitly named in the following code cell; setup-only cells emit no file.

Method: Apply the scientific definition stated above, enforce its declared dimensions/counts/parameters, validate any temporary output, then atomically install it without modifying a source file.


In [ ]:
paths = [PREPROCESSED_ROOT / "MERRA2_Processed" / "Z3" / f"MERRA2.Z3.{year}.nc" for year in range(1980, 2026)]
missing = [str(path) for path in paths if not path.exists()]
if missing:
    raise FileNotFoundError(f"MERRA-2 Z3 1980--2025 incomplete: {missing[:3]}")
merra_zonal = xr.concat([pressure_zonal_z3(path, hybrid=False) for path in paths], dim="time").sortby("time")
years = np.asarray(merra_zonal.date.values) // 10000
merra_reference = train_fixed_nam_reference(
    merra_zonal.isel(time=np.flatnonzero((years >= 1980) & (years <= 2000)))
)
merra_reference.attrs.update(
    product_version=PRODUCT_VERSION, reference_period="1980-01-01--2000-12-31",
    source_period="1980-01-01--2025-12-31",
)
write_netcdf_atomic(
    merra_reference, product_path("nam", "merra2_fixed_eof_reference.nc"),
    required_vars={
        "eof1_weighted": ("plev", "lat"), "monthly_pc_std": ("plev",),
        "explained_variance_fraction": ("plev",),
        "valid_monthly_sample_count": ("plev",),
        "daily_height_climatology": ("month_day", "plev", "lat"),
    }, required_coords=("plev", "lat", "month_day"), exact_sizes={"month_day": 365},
    overwrite=OVERWRITE,
)
merra_nam = project_nam(merra_zonal, merra_reference)
merra_nam.attrs.update(
    product_version=PRODUCT_VERSION, source_period="1980--2025",
    eof_training="calendar-month anomalies, 1980--2000",
)
write_nam(merra_nam, "merra2_daily_nam_ao.nc")


## WACCM 210-year reference and pressure-source projection

Inputs: all 210 staged hybrid LONGRUN Z3 source years, the strict 207+23 ozone ranking, the audited 209/23 pressure-year manifests, and staged pressure-level Z3. Outputs: one EOF trained on the complete 210-year LONGRUN integration plus LONGRUN/BWCN daily NAM/AO projected for every audited pressure-source year. Method: form each natural-calendar monthly mean from its actual finite daily values, record the valid monthly sample count at every pressure, and keep EOF training independent of ozone-spring completeness. Predecessor-only pressure years are projected so October--December can attach to a ranked successor; BWCN/year 0008 use this unchanged fixed reference.


In [ ]:
manifest = pd.read_csv(product_path("ozone", "waccm_event_manifest.csv"))

def pressure_year_contract(segment, expected_raw, expected_ranking, expected_pressure):
    path = PREPROCESSED_ROOT / "manifests" / f"waccm_{segment.lower()}_pressure_years.tsv"
    table = pd.read_csv(path, sep="\t")
    required = {
        "segment", "pressure_year", "raw_available", "ranking_event_complete",
        "needed_as_predecessor", "pressure_source_required",
    }
    if required - set(table):
        raise ValueError(f"{path}: missing columns {sorted(required - set(table))}")
    if set(table.segment.astype(str)) != {segment}:
        raise RuntimeError(f"{path}: segment column is not exactly {segment}")
    if table.pressure_year.astype(int).duplicated().any():
        raise RuntimeError(f"{path}: duplicate pressure years")
    ranking_years = set(
        manifest.loc[manifest.source_segment.astype(str) == segment, "model_year"].astype(int)
    )
    audited_ranking = set(
        table.loc[table.ranking_event_complete.astype(int) == 1, "pressure_year"].astype(int)
    )
    pressure_years = set(
        table.loc[table.pressure_source_required.astype(int) == 1, "pressure_year"].astype(int)
    )
    predecessor_years = set(
        table.loc[table.needed_as_predecessor.astype(int) == 1, "pressure_year"].astype(int)
    )
    raw_years = set(
        table.loc[table.raw_available.astype(int) == 1, "pressure_year"].astype(int)
    )
    if ranking_years != audited_ranking:
        raise RuntimeError(f"{segment}: ozone ranking and pressure-year manifest differ")
    if pressure_years != ranking_years | predecessor_years:
        raise RuntimeError(f"{segment}: pressure-source union is inconsistent")
    if (
        len(raw_years) != expected_raw
        or len(ranking_years) != expected_ranking
        or len(pressure_years) != expected_pressure
    ):
        raise RuntimeError(
            f"{segment}: raw/ranking/pressure counts="
            f"{(len(raw_years), len(ranking_years), len(pressure_years))}; "
            f"expected={(expected_raw, expected_ranking, expected_pressure)}"
        )
    selected = table.loc[table.pressure_source_required.astype(int) == 1]
    if not (selected.raw_available.astype(int) == 1).all():
        raise RuntimeError(f"{segment}: required pressure source is not raw-available")
    return raw_years, ranking_years, pressure_years, path

def segment_zonal(segment, z_root, wanted, *, hybrid):
    paths = sorted(Path(z_root).glob("*.Z3.nc"))
    parsed_years = [parse_year(path) for path in paths]
    if len(parsed_years) != len(set(parsed_years)):
        raise RuntimeError(f"{segment}: duplicate staged pressure-level Z3 years")
    by_year = dict(zip(parsed_years, paths))
    missing = sorted(wanted - set(by_year))
    if missing:
        raise FileNotFoundError(f"{segment} Z3 missing requested source years: {missing[:3]}")
    arrays = []
    for year in sorted(wanted):
        field = pressure_zonal_z3(by_year[year], hybrid=hybrid)
        dates = np.asarray(field.date.values, dtype=int)
        if (
            dates.size != 365
            or np.unique(dates).size != 365
            or not np.all(dates // 10000 == year)
            or np.any(((dates // 100) % 100 == 2) & (dates % 100 == 29))
        ):
            raise RuntimeError(f"{segment} {year:04d}: expected one exact 365-day no-leap year")
        field = field.assign_coords(model_year=("time", np.full(field.sizes["time"], year)))
        arrays.append(field)
    return xr.concat(arrays, dim="time")

long_raw_years, long_ranked_years, long_pressure_years, long_pressure_manifest = pressure_year_contract(
    "LONGRUN", 210, 207, 209
)
long_training = segment_zonal(
    "LONGRUN", PREPROCESSED_ROOT / "B2000WCN001002_timefixed" / "Z3",
    long_raw_years, hybrid=True,
)
if np.unique(long_training.model_year).size != 210:
    raise RuntimeError("WACCM NAM reference requires all 210 raw LONGRUN years")
waccm_reference = train_fixed_nam_reference(long_training)
waccm_reference.attrs.update(
    product_version=PRODUCT_VERSION, source_segment="LONGRUN",
    training_population="complete 210-year LONGRUN integration; independent of O3 completeness",
    training_model_year_count=210,
    training_source_years=",".join(f"{year:04d}" for year in sorted(long_raw_years)),
    projection_pressure_source_year_count=209,
    pressure_year_manifest=str(long_pressure_manifest),
    padding_projection_years=",".join(
        f"{year:04d}" for year in sorted(long_pressure_years - long_ranked_years)
    ),
)
write_netcdf_atomic(
    waccm_reference, product_path("nam", "waccm_fixed_eof_reference.nc"),
    required_vars={
        "eof1_weighted": ("plev", "lat"), "monthly_pc_std": ("plev",),
        "explained_variance_fraction": ("plev",),
        "valid_monthly_sample_count": ("plev",),
        "daily_height_climatology": ("month_day", "plev", "lat"),
    }, required_coords=("plev", "lat", "month_day"), exact_sizes={"month_day": 365},
    overwrite=OVERWRITE,
)
del long_training
long_zonal = segment_zonal(
    "LONGRUN",
    PREPROCESSED_ROOT / "B2000WCN001002_timefixed" / "interpolated" / "Z3",
    long_pressure_years, hybrid=False,
)
if np.unique(long_zonal.model_year).size != 209:
    raise RuntimeError("WACCM LONGRUN NAM projection requires 209 pressure-source years")
long_nam = project_nam(long_zonal, waccm_reference).assign_coords(model_year=long_zonal.model_year)
long_nam.attrs.update(
    product_version=PRODUCT_VERSION, source_segment="LONGRUN",
    eof_training_source_year_count=210, pressure_source_year_count=209,
    pressure_year_manifest=str(long_pressure_manifest),
    padding_projection_years=",".join(
        f"{year:04d}" for year in sorted(long_pressure_years - long_ranked_years)
    ),
)
write_nam(long_nam, "waccm_longrun_daily_nam_ao.nc")

bwcn_raw_years, bwcn_ranked_years, bwcn_pressure_years, bwcn_pressure_manifest = pressure_year_contract(
    "BWCN", 24, 23, 23
)
bwcn_zonal = segment_zonal(
    "BWCN", PREPROCESSED_ROOT / "BWCN" / "interpolated" / "Z3",
    bwcn_pressure_years, hybrid=False,
)
if np.unique(bwcn_zonal.model_year).size != 23:
    raise RuntimeError("BWCN NAM projection requires all 23 pressure-source years")
bwcn_nam = project_nam(bwcn_zonal, waccm_reference).assign_coords(model_year=bwcn_zonal.model_year)
bwcn_nam.attrs.update(
    product_version=PRODUCT_VERSION, source_segment="BWCN",
    eof_training_source_year_count=210, pressure_source_year_count=23,
    pressure_year_manifest=str(bwcn_pressure_manifest),
    padding_projection_years=",".join(
        f"{year:04d}" for year in sorted(bwcn_pressure_years - bwcn_ranked_years)
    ),
)
write_nam(bwcn_nam, "waccm_bwcn_daily_nam_ao.nc")

target = bwcn_nam.where(bwcn_nam.model_year == 8, drop=True)
target.attrs.update(product_version=PRODUCT_VERSION, reference_event="BWCN year 0008")
write_nam(target, "waccm_bwcn_year0008_nam_ao.nc")


## January, February, and March member NAM/AO

All 30 members per case are directly projected onto the same fixed LONGRUN EOF and standardized by its monthly-PC standard deviation. March has no February-derived empirical calibration.

Inputs: The read-only archive or schema-validated staging paths explicitly opened in the following code cell.

Outputs: The in-memory object(s) and atomic staging product(s) explicitly named in the following code cell; setup-only cells emit no file.

Method: Apply the scientific definition stated above, enforce its declared dimensions/counts/parameters, validate any temporary output, then atomically install it without modifying a source file.


In [ ]:
def member_nam(case, root, *, staged_pressure_levels):
    candidates = (
        sorted((root / "interpolated" / "Z3").glob("*.Z3.nc"))
        if staged_pressure_levels else sorted(root.glob("*.nc"))
    )
    if len(candidates) != 30:
        raise RuntimeError(f"{case}: expected 30 Z3 members; found {len(candidates)}")
    arrays, labels, common_date = [], [], None
    for path in candidates:
        zonal = pressure_zonal_z3(path, hybrid=False)
        projected = project_nam(zonal, waccm_reference)
        dates = np.asarray(projected.date.values, dtype=np.int32)
        if common_date is None:
            common_date = dates
        elif not np.array_equal(common_date, dates):
            raise RuntimeError(f"{case}: member calendars differ")
        arrays.append(projected.drop_vars("date"))
        labels.append(parse_member(path))
    combined = xr.concat(arrays, dim=xr.IndexVariable("member", labels))
    combined = combined.assign_coords(date=("time", common_date))
    combined.attrs.update(
        product_version=PRODUCT_VERSION, case=case, member_count=30,
        projection="direct fixed LONGRUN EOF", calibration="none",
        ao_definition="nam at exactly 1000 hPa",
    )
    write_nam(combined, f"hindcast_{case}_nam_ao.nc", member_product=True)

member_nam("0008-01", PREPROCESSED_ROOT / "Hindcast" / "0008-01", staged_pressure_levels=True)
member_nam("0008-02", PREPROCESSED_ROOT / "Hindcast" / "0008-02", staged_pressure_levels=True)
member_nam("0008-03", MARINA_ROOT, staged_pressure_levels=False)
